# पाठ 11 - एजेंट-टू-एजेंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकॉल क्या है?

The **एजेंट-टू-एजेंट (A2A) प्रोटोकॉल** एक खुला मानक है जो AI एजेंटों को संवाद करने, एक-दूसरे को खोजने, और सहयोग करने में सक्षम बनाता है — भले ही वे विभिन्न फ्रेमवर्क पर बने हों या विभिन्न सेवाओं द्वारा होस्ट किए गए हों।

Key concepts:

- **खोज** – एजेंट अपनी क्षमताओं का वर्णन करने वाला *एजेंट कार्ड* प्रकाशित करते हैं, जिससे अन्य एजेंटों (या ऑर्केस्ट्रेटर) के लिए किसी कार्य के लिए सही विशेषज्ञ ढूँढना आसान हो जाता है।
- **संदेश प्रेषण** – एजेंट एक सामान्य प्रोटोकॉल के माध्यम से संरचित संदेशों का आदान-प्रदान करते हैं, ताकि एक एजेंट का अनुरोध दूसरी एजेंट द्वारा उसकी आंतरिक कार्यान्वयन की परवाह किए बिना समझा और पूरा किया जा सके।
- **कार्य जीवनचक्र** – A2A *सबमिट किया गया*, *कार्यरत*, *पूरा हुआ*, और *विफल* जैसी अवस्थाओं को परिभाषित करता है, जिससे ऑर्केस्ट्रेटर को यह पूरी दृश्यता मिलती है कि सौंपा गया कार्य कैसे प्रगति कर रहा है।

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## विशेषीकृत यात्रा एजेंट बनाना


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ़्लो के माध्यम से बहु-एजेंट सहयोग

हम तीनों एजेंटों को एक क्रमिक वर्कफ़्लो में जोड़ते हैं जो A2A संदेश पारित करने का अनुकरण करता है:

1. **CurrencyExchangeAgent** उपयोगकर्ता के अनुरोध को प्राप्त करता है और मुद्रा संबंधी मार्गदर्शन प्रदान करता है.
2. **ActivityPlannerAgent** समृद्ध संदर्भ को प्राप्त करता है और गतिविधि सिफारिशें जोड़ता है.
3. **TravelManagerAgent** दोनों इनपुटों को एकीकृत करके अंतिम यात्रा सार बनाता है.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादन में A2A को समझना

उत्पादन वातावरण में A2A प्रोटोकॉल शक्तिशाली क्रॉस-सर्विस परिदृश्यों को सक्षम करता है:

| Capability | Description |
|---|---|
| **क्रॉस-फ्रेमवर्क इंटरऑपरेबिलिटी** | एक फ्रेमवर्क से बनाया गया एक एजेंट किसी अन्य किसी भी A2A-compliant फ्रेमवर्क से बने एजेंट को कार्य सौंप सकता है, जिससे वास्तविक क्रॉस-ऑर्गनाइज़ेशन इंटरऑपरेबिलिटी सक्षम होती है। |
| **सेवा सीमाएँ** | एजेंट अलग-अलग माइक्रोसर्विसेज़, क्लाउड क्षेत्रों, या यहां तक कि विभिन्न संगठनों में रह सकते हैं, फिर भी वे निर्बाध रूप से सहयोग कर सकते हैं। |
| **डायनामिक डिस्कवरी** | एक ऑर्केस्ट्रेटर रनटाइम पर किसी Agent Card रजिस्ट्री से क्वेरी कर सकता है ताकि किसी दिए गए उप-कार्य के लिए सबसे उपयुक्त विशेषज्ञ मिल सके। |
| **स्ट्रीमिंग और पुश सूचनाएँ** | A2A वास्तविक समय की प्रगति अपडेट्स के लिए Server-Sent Events (SSE) और लंबी अवधि के कार्यों के लिए पुश सूचनाओं का समर्थन करता है। |

The workflow we built above is a simplified, in-process version of this pattern. In a real
परिनियोजन में प्रत्येक एजेंट एक HTTP endpoint उपलब्ध कराएगा, एक Agent Card प्रकाशित करेगा, और संचार करेगा
A2A JSON-RPC प्रोटोकॉल के माध्यम से।


## सारांश

इस पाठ में आपने सीखा:

1. **A2A प्रोटोकॉल क्या है** — एजेंट-से-एजेंट खोज, मैसेजिंग,
   और कार्य प्रबंधन.
2. **विशेषीकृत एजेंट कैसे बनाएँ** — एक Currency Exchange एजेंट, एक Activity Planner एजेंट,
   और एक Travel Manager ऑर्केस्ट्रेटर.
3. **एजेंटों को वर्कफ़्लो में जोड़ने का तरीका** — `WorkflowBuilder` का उपयोग करके अनुक्रमिक
   एजेंटों के बीच संदेश पासिंग को मॉडल करने के लिए.
4. **A2A उत्पादन में कैसे काम करता है** — क्रॉस-फ़्रेमवर्क, क्रॉस-सर्विस सहयोग सक्षम करना
   गतिशील खोज और स्ट्रीमिंग अपडेट के साथ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। हालांकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान रखें कि स्वचालित अनुवादों में त्रुटियाँ या inaccuracies हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफ़हमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
